# NB03: Data Analysis

**Author:** martinezmerino

This notebook explores the tidy team-match table from NB02 to answer: **is home advantage equally strong across Europe's big five leagues, or does it vary?** Analysis is exploratory - I'm comparing groups and checking consistency, not fitting a predictive or inferential model.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

df = pd.read_csv("data/processed/matches_team_level.csv")
df.head()

,league,league_code,season,match_id,date,matchday,team,opponent,venue,goals_for,goals_against,goal_diff,result,points
0,Premier League,PL,2025,537785,2025-08-15,1,Liverpool FC,AFC Bournemouth,home,4,2,2,win,3
1,Premier League,PL,2025,537785,2025-08-15,1,AFC Bournemouth,Liverpool FC,away,2,4,-2,loss,0
2,Premier League,PL,2025,537786,2025-08-16,1,Aston Villa FC,Newcastle United FC,home,0,0,0,draw,1
3,Premier League,PL,2025,537786,2025-08-16,1,Newcastle United FC,Aston Villa FC,away,0,0,0,draw,1
4,Premier League,PL,2025,537787,2025-08-16,1,Brighton & Hove Albion FC,Fulham FC,home,1,1,0,draw,1


## Checking the comparison is fair before I start

Before comparing home and away performance, I check that every team played the same number of home and away matches. If a team had more home fixtures than away ones, it could look like it "benefits" from home advantage just because of the schedule, not because it actually performs differently at home.

In [2]:
venue_counts = df.groupby(["league", "team", "venue"]).size().unstack("venue")
venue_counts["imbalance"] = (venue_counts["home"] - venue_counts["away"]).abs()

print("Teams with unequal home/away match counts:")
print(venue_counts[venue_counts["imbalance"] > 0])
print(f"\nMax home/away imbalance across all {len(venue_counts)} teams: {venue_counts['imbalance'].max()}")

Teams with unequal home/away match counts:
venue                away  home  imbalance
league  team                              
Ligue 1 FC Nantes      17    16          1
        Toulouse FC    16    17          1

Max home/away imbalance across all 96 teams: 1


Two teams, FC Nantes and Toulouse FC (Ligue 1), are off by one match. Checking the full match list for that fixture (not just the `FINISHED` ones I originally pulled) shows why: their matchday-34 meeting was decided administratively (`status: AWARDED`, not an on-pitch result), while the first leg was played normally and finished 2-2. Since an awarded match doesn't reflect what happened on the pitch, excluding it from a home-advantage analysis is the right call - it just means these two teams have 33 matches instead of 34, a negligible imbalance out of 96 teams. So the comparison below is fair for effectively all of them.

## Defining "home advantage"

I measure home advantage per team as the gap in points per game (PPG) between its home fixtures and its away fixtures: `home_ppg - away_ppg`. A team with a gap of zero gets no measurable benefit from playing at home; a large positive gap means home matches are systematically more rewarding than away ones.

I compute this per team first, and only then summarise by league, so I can check whether a league's average gap is broad-based across most of its teams or driven by a handful of outliers - a check I skipped on my midterm and had to add after the fact.

In [3]:
team_venue = (
    df.groupby(["league", "team", "venue"])
    .agg(
        matches=("points", "size"),
        points=("points", "sum"),
        goal_diff=("goal_diff", "sum"),
        wins=("result", lambda s: (s == "win").sum()),
    )
    .reset_index()
)
team_venue["ppg"] = team_venue["points"] / team_venue["matches"]
team_venue["win_rate"] = team_venue["wins"] / team_venue["matches"]
team_venue["goal_diff_per_game"] = team_venue["goal_diff"] / team_venue["matches"]

team_wide = team_venue.pivot(index=["league", "team"], columns="venue", values=["ppg", "win_rate", "goal_diff_per_game"])
team_wide.columns = [f"{stat}_{venue}" for stat, venue in team_wide.columns]
team_wide = team_wide.reset_index()

team_wide["ppg_gap"] = team_wide["ppg_home"] - team_wide["ppg_away"]
team_wide["win_rate_gap"] = team_wide["win_rate_home"] - team_wide["win_rate_away"]
team_wide["goal_diff_gap"] = team_wide["goal_diff_per_game_home"] - team_wide["goal_diff_per_game_away"]

team_wide.sort_values("ppg_gap", ascending=False).head(10)

,league,team,ppg_away,ppg_home,win_rate_away,win_rate_home,goal_diff_per_game_away,goal_diff_per_game_home,ppg_gap,win_rate_gap,goal_diff_gap
22,La Liga,Elche CF,0.421053,1.842105,0.052632,0.473684,-1.000000,0.578947,1.421053,0.421053,1.578947
29,La Liga,RCD Mallorca,0.473684,1.736842,0.105263,0.473684,-1.052632,0.526316,1.263158,0.368421,1.578947
20,La Liga,Club Atlético de Madrid,1.210526,2.421053,0.315789,0.789474,-0.210526,1.157895,1.210526,0.473684,1.368421
19,La Liga,CA Osasuna,0.526316,1.684211,0.105263,0.473684,-0.684211,0.368421,1.157895,0.368421,1.052632
23,La Liga,FC Barcelona,1.947368,3.000000,0.631579,1.000000,0.631579,2.473684,1.052632,0.368421,1.842105
37,La Liga,Villarreal CF,1.368421,2.421053,0.368421,0.789474,-0.157895,1.526316,1.052632,0.421053,1.684211
13,Bundesliga,SC Freiburg,0.882353,1.882353,0.235294,0.529412,-1.000000,0.647059,1.000000,0.294118,1.647059
65,Premier League,Fulham FC,0.894737,1.842105,0.210526,0.578947,-0.736842,0.526316,0.947368,0.368421,1.263158
16,Bundesliga,VfB Stuttgart,1.352941,2.294118,0.352941,0.705882,0.470588,0.823529,0.941176,0.352941,0.352941
66,Premier League,Leeds United FC,0.789474,1.684211,0.105263,0.473684,-0.789474,0.421053,0.894737,0.368421,1.210526


## A first look: who has the biggest home-away gap?

Six of the top 10 teams by PPG gap play in La Liga (Elche CF +1.42, RCD Mallorca +1.26, Atlético Madrid +1.21, CA Osasuna +1.16, FC Barcelona +1.05, Villarreal CF +1.05), which already points toward La Liga having the strongest overall effect - consistent with what the league summary below confirms with real averages, not just a top-10 list.

What's more interesting is *which* La Liga teams these are. I pulled each team's overall season PPG (home and away combined) to see where they actually finished: Barcelona were 1st, Villarreal 3rd and Atlético 4th, but Elche were 15th, Osasuna 16th and Mallorca 18th out of 20. So the biggest home-advantage gaps aren't just a "strong teams win everywhere" effect - a mid/bottom-table side (Elche) tops the whole list, right alongside the league champions. I check this more systematically further down.

## Does the gap hold across most teams in each league, or just a few?

In [4]:
league_summary = (
    team_wide.groupby("league")
    .agg(
        teams=("team", "size"),
        mean_ppg_gap=("ppg_gap", "mean"),
        median_ppg_gap=("ppg_gap", "median"),
        teams_favoured_at_home=("ppg_gap", lambda s: (s > 0).sum()),
        teams_favoured_away=("ppg_gap", lambda s: (s < 0).sum()),
    )
    .reset_index()
)
league_summary["pct_teams_favoured_at_home"] = 100 * league_summary["teams_favoured_at_home"] / league_summary["teams"]
league_summary.sort_values("mean_ppg_gap", ascending=False)

,league,teams,mean_ppg_gap,median_ppg_gap,teams_favoured_at_home,teams_favoured_away,pct_teams_favoured_at_home
1,La Liga,20,0.671053,0.631579,19,1,95.000000
2,Ligue 1,18,0.498366,0.558824,17,1,94.444444
3,Premier League,20,0.378947,0.526316,15,4,75.000000
0,Bundesliga,18,0.362745,0.411765,15,3,83.333333
4,Serie A,20,0.118421,0.131579,13,6,65.000000


## Answering the question directly

Ranked by mean gap: **La Liga (0.67) > Ligue 1 (0.50) > Premier League (0.38) > Bundesliga (0.36) > Serie A (0.12)**. La Liga and Ligue 1 are also the most *broad-based* - 95% and 94% of their teams are favoured at home respectively, so the average isn't being pulled up by a handful of outliers. Serie A is the weakest on both counts: the smallest average gap **and** the smallest share of teams favoured at home (65%, or 13 of 20).

One thing the ranking-by-mean hides: **Bundesliga has a slightly lower average gap than the Premier League (0.36 vs 0.38), but a noticeably higher share of teams favoured at home (83% vs 75%)**. That means Bundesliga's home advantage, while a bit smaller on average, is more evenly shared across its clubs, while the Premier League's average is propped up more by a subset of teams with a large gap and dragged down by others with none. A single average, on its own, would have hidden this - which is exactly why I computed both.

## Which teams actually do *worse* at home?

In [5]:
negative_gap = team_wide[team_wide["ppg_gap"] < 0].sort_values("ppg_gap")[["league", "team", "ppg_gap"]]
print(f"{len(negative_gap)} of 96 teams have a negative home-away PPG gap:\n")
print(negative_gap.to_string(index=False))
print()
print(negative_gap["league"].value_counts())

15 of 96 teams have a negative home-away PPG gap:

        league                 team   ppg_gap
       Serie A      Bologna FC 1909 -0.631579
Premier League Tottenham Hotspur FC -0.578947
    Bundesliga        VfL Wolfsburg -0.529412
       La Liga     RC Celta de Vigo -0.421053
    Bundesliga      1. FSV Mainz 05 -0.352941
       Serie A             AC Milan -0.315789
       Serie A     Hellas Verona FC -0.263158
Premier League Nottingham Forest FC -0.210526
       Serie A       Udinese Calcio -0.210526
       Ligue 1            Lille OSC -0.176471
Premier League           Everton FC -0.157895
Premier League    Crystal Palace FC -0.157895
       Serie A    Parma Calcio 1913 -0.157895
       Serie A         US Cremonese -0.105263
    Bundesliga    FC Bayern München -0.058824

league
Serie A           6
Premier League    4
Bundesliga        3
La Liga           1
Ligue 1           1
Name: count, dtype: int64


15 of the 96 teams (16%) actually picked up fewer points per game at home than away this season. Six of those 15 play in Serie A - more than any other league, and consistent with Serie A already being the weakest league for home advantage overall. The two most extreme cases are Bologna FC 1909 (-0.63) and Tottenham Hotspur FC (-0.58), the two biggest home *disadvantages* across all five leagues. At the other end, FC Bayern München's gap is only -0.06 - close enough to zero that it looks more like a team that's simply dominant everywhere, home or away, rather than a real home disadvantage.

I don't have crowd size, referee, or travel data in this dataset, so I can't say *why* these specific teams buck the trend - I'm reporting them as evidence that "home advantage" is a strong league-wide tendency, not a universal rule, rather than trying to explain each case.

## Chart 1: the home-advantage gap, team by team

A bar chart of the league averages would hide how spread out teams are within each league, so I plot every team's individual gap as a point (a strip plot), with a diamond marking each league's mean. This shows both the central tendency and how consistent - or not - the effect is.

In [6]:
fig1 = px.strip(
    team_wide,
    x="league",
    y="ppg_gap",
    color="league",
    hover_data=["team"],
    title="Home advantage (home PPG minus away PPG) by team, big five leagues 2025-26",
)
fig1.add_hline(y=0, line_dash="dash", line_color="gray")

for league, row in league_summary.set_index("league").iterrows():
    fig1.add_scatter(
        x=[league],
        y=[row["mean_ppg_gap"]],
        mode="markers",
        marker=dict(symbol="diamond", size=14, color="black"),
        name=f"{league} mean",
        showlegend=False,
    )

fig1.update_layout(yaxis_title="Home PPG - Away PPG", xaxis_title=None, showlegend=False)
fig1.write_image(str(FIGURES_DIR / "ppg_gap_by_league.png"), width=900, height=550, scale=2)
fig1.show()

## What the spread shows

The range each league covers (min to max team gap) tells a story the average alone can't: La Liga spans -0.42 to +1.42, the widest range of the five - it has both a strong average *and* some real extremes. Serie A spans only -0.63 to +0.58: not a single Serie A team clears +0.6, while every other league has at least one team above +0.8. Ligue 1 has the narrowest low end (-0.18 to +0.88) - no Ligue 1 team is badly disadvantaged at home, which lines up with it having the second-highest share of teams favoured at home (94%).

## Chart 2: is the points gap backed up by goals, or just lucky results?

If home advantage is real rather than a quirk of close results, a team's gap in points per game should line up with its gap in goal difference per game. I check this per team, across all five leagues at once.

In [7]:
correlation = team_wide[["ppg_gap", "goal_diff_gap"]].corr().iloc[0, 1]
print(f"Correlation between PPG gap and goal-difference-per-game gap: {correlation:.2f}")

fig2 = px.scatter(
    team_wide,
    x="goal_diff_gap",
    y="ppg_gap",
    color="league",
    hover_data=["team"],
    title="Points gap vs goal-difference gap (home minus away), per team",
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray")
fig2.add_vline(x=0, line_dash="dash", line_color="gray")
fig2.update_layout(xaxis_title="Goal-difference-per-game gap (home - away)", yaxis_title="PPG gap (home - away)")
fig2.write_image(str(FIGURES_DIR / "ppg_gap_vs_goal_diff_gap.png"), width=900, height=550, scale=2)
fig2.show()

Correlation between PPG gap and goal-difference-per-game gap: 0.82


## Does that relationship hold within every league, or just when I pool them all together?

A pooled correlation across five leagues can be misleading if the leagues themselves differ in how points and goals relate - I check the same correlation separately within each league before trusting the pooled 0.82 figure.

In [8]:
per_league_corr = (
    team_wide.groupby("league")
    .apply(lambda g: g["ppg_gap"].corr(g["goal_diff_gap"]))
    .reset_index(name="correlation")
    .sort_values("correlation", ascending=False)
)
per_league_corr

,league,correlation
3,Premier League,0.900747
1,La Liga,0.893081
2,Ligue 1,0.807672
4,Serie A,0.753243
0,Bundesliga,0.465316


The pooled 0.82 holds up reasonably well league by league, but not equally: Premier League (0.90), La Liga (0.89) and Ligue 1 (0.81) are all close to or above the pooled figure, Serie A (0.75) a little below it, but **Bundesliga sits at only 0.47** - noticeably weaker than the rest. In the Bundesliga specifically, a team's points advantage at home is a much less reliable signal of its goalscoring advantage at home than in the other four leagues; something else (tighter margins, more draws, or just a smaller league with more noise per team) is doing more of the work there. I don't have enough evidence in this dataset to say which - I'm flagging it rather than smoothing it over with the pooled number.

## Is home advantage just bigger for stronger teams?

The top-10 list earlier already hinted this might not be true (Elche CF, a bottom-half team, topped it). I check this properly by using each team's overall season PPG (home and away combined) as a simple quality proxy, and correlating it with the home-away PPG gap.

In [9]:
overall_quality = df.groupby(["league", "team"])["points"].mean().reset_index(name="overall_ppg")
quality_vs_gap = team_wide.merge(overall_quality, on=["league", "team"])

pooled_quality_corr = quality_vs_gap["overall_ppg"].corr(quality_vs_gap["ppg_gap"])
print(f"Pooled correlation, overall PPG vs home-away PPG gap: {pooled_quality_corr:.2f}")

per_league_quality_corr = (
    quality_vs_gap.groupby("league")
    .apply(lambda g: g["overall_ppg"].corr(g["ppg_gap"]))
    .reset_index(name="correlation")
    .sort_values("correlation", ascending=False)
)
per_league_quality_corr

Pooled correlation, overall PPG vs home-away PPG gap: 0.19


,league,correlation
3,Premier League,0.341413
4,Serie A,0.233772
2,Ligue 1,0.206391
1,La Liga,0.199278
0,Bundesliga,0.071025


This one didn't confirm the hunch the top-10 list raised. The pooled correlation between overall team quality and the home-away gap is weak (0.19), and it stays weak in every single league (from 0.07 in the Bundesliga up to 0.34 in the Premier League - still a fairly loose relationship). So being a stronger team, in general, is not strongly associated with getting a bigger boost from playing at home. Combined with La Liga's top-10 list mixing 1st-place Barcelona with 15th-to-18th-placed Elche, Osasuna and Mallorca, the evidence points toward home advantage operating largely independently of how good a team is overall - I'd rather report that honestly than force a "strong teams benefit most" story the data doesn't really support.

## Results summary

In [10]:
summary = league_summary.sort_values("mean_ppg_gap", ascending=False).reset_index(drop=True)
for _, row in summary.iterrows():
    print(
        f"{row['league']}: mean home-away PPG gap = {row['mean_ppg_gap']:.2f}, "
        f"{row['teams_favoured_at_home']:.0f}/{row['teams']:.0f} teams favoured at home "
        f"({row['pct_teams_favoured_at_home']:.0f}%)"
    )

print(f"\nOverall correlation between PPG gap and goal-diff gap: {correlation:.2f}")
print(f"League with strongest home advantage: {summary.iloc[0]['league']}")
print(f"League with weakest home advantage: {summary.iloc[-1]['league']}")

La Liga: mean home-away PPG gap = 0.67, 19/20 teams favoured at home (95%)
Ligue 1: mean home-away PPG gap = 0.50, 17/18 teams favoured at home (94%)
Premier League: mean home-away PPG gap = 0.38, 15/20 teams favoured at home (75%)
Bundesliga: mean home-away PPG gap = 0.36, 15/18 teams favoured at home (83%)
Serie A: mean home-away PPG gap = 0.12, 13/20 teams favoured at home (65%)

Overall correlation between PPG gap and goal-diff gap: 0.82
League with strongest home advantage: La Liga
League with weakest home advantage: Serie A


## Sources for the physical/behavioural reasoning

This dataset only has match results, so it can show *that* home advantage varies by league, not directly test *why*. The candidate mechanisms discussed in the sports-science literature are:

- Wikipedia, *Home advantage*: https://en.wikipedia.org/wiki/Home_advantage - overview of the commonly cited factors: crowd support, travel fatigue, familiarity with the pitch, and referee bias under crowd pressure.
- Pollard, R. (1986). "Home advantage in soccer: A retrospective analysis." *Journal of Sports Sciences*, 4(3), 237-248. The foundational academic study establishing that home advantage in football is systematic rather than random.
- Dohmen, T. J. (2008). "The influence of social forces: Evidence from the behavior of football referees." *Economic Inquiry*, 46(3), 411-424. Finds that referees in Germany's Bundesliga award more stoppage time and marginal decisions to home teams, more strongly when there's a running track separating fans from the pitch versus stands right on top of the action - i.e. crowd proximity affecting officiating.

I can't test crowd size, referee decisions, or travel distance directly with this dataset (unlike the midterm, where I had humidity data to check one candidate mechanism), so I'm citing these as context for *why* a gap this size and this variable-by-league is plausible, not as proof of which mechanism drives it in my data specifically.

## Conclusions

**Main finding:** home advantage is real and broad-based across Europe's big five leagues in 2025-26, but its strength differs sharply by league. La Liga shows the strongest effect (mean home-away PPG gap of +0.67, with 95% of its teams favoured at home) and Serie A the weakest (+0.12, only 65% of teams favoured, and not a single team clearing +0.6).

**What I checked and didn't just assume:**
- The home/away schedule is fair - only 2 of 96 teams have any imbalance at all, and I confirmed with the raw API data that it's because one fixture was administratively awarded rather than played, not a data error.
- A league's average gap can hide how broad-based it is: Bundesliga has a *lower* mean gap than the Premier League (0.36 vs 0.38) but a *higher* share of teams favoured at home (83% vs 75%) - the average alone would have missed this.
- I named the 15 teams (of 96) with an actual home disadvantage rather than only reporting the leagues where the trend holds; six of them play in Serie A, reinforcing that it's the weakest league for the effect.
- The points gap is backed up by the goal-difference gap (pooled r = 0.82), suggesting real on-pitch performance rather than lucky results - but this weakens substantially in the Bundesliga specifically (r = 0.47), which the pooled figure alone would have hidden.
- I tested whether home advantage is really just a "strong teams win everywhere" effect using each team's overall season PPG as a quality proxy, and found only a weak relationship (pooled r = 0.19, weak in every league individually) - a finding that didn't confirm my initial hunch from the top-10 list, and I'm reporting it as such rather than dropping it.

**Limitations:** this covers a single season (2025-26), so I can't say whether these league rankings on home advantage are stable year to year or just this season's pattern. I don't have crowd size, referee-decision, or travel-distance data, so while the sports-science literature points to plausible mechanisms (crowd support, referee bias, travel fatigue, pitch familiarity), I can't test which of them is actually driving the gap in this specific dataset - only that the size and variability of the effect is consistent with those mechanisms being real.